# Task 6 — losowy eksperyment na repozytorium N=200

Eksperyment wykonuje co najmniej 10 operacji instalacji i usuwania na repozytorium grupy `k12`.

- Repozytorium: `data/repos/repos/k12/repo_200.json`
- Ziarno generatora: **42** (`random.Random(42)`)
- Początkowy stan: pusty; stan pozostaje w pamięci notebooka i nie zapisuje `state.txt`.
- Każdy stan po operacji jest sprawdzany przez `verify` z zadania 4.4.

Uruchom notebook z katalogu `Package_Manager/Project`.

In [1]:
from pathlib import Path
import random
import sys
import time
from contextlib import redirect_stdout
from io import StringIO
import pandas as pd

GROUP = "k12"
N = 200
SEED = 42
PROJECT_DIR = Path.cwd()
if not (PROJECT_DIR / "manager.py").is_file():
    raise RuntimeError("Run this notebook from Package_Manager/Project")
SRC = PROJECT_DIR / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

import z3
from package_manager.domain.errors import PackageError
from package_manager.domain.model import State
from package_manager.domain.verification import verify
from package_manager.infrastructure.parsing import load_repository
from package_manager.use_cases.install import install
from package_manager.use_cases.install_with_state import build_change_plan
from package_manager.use_cases.remove import remove

REPO_PATH = PROJECT_DIR / "data" / "repos" / "repos" / GROUP / f"repo_{N}.json"
repo = load_repository(REPO_PATH)

# Bare requests for these packages are known to resolve dependencies in this repository.
DEPENDENCY_ROOTS = ["aiohttp", "h2", "boto3"]
safe_install_candidates = sorted(
    name
    for name, package in repo.packages.items()
    if any(not info.requires and not info.conflicts for info in package.versions.values())
)
assert all(name in repo.packages for name in DEPENDENCY_ROOTS)
print(f"Repository: {REPO_PATH.relative_to(PROJECT_DIR)}")
print(f"Group: {GROUP}; N={N}; seed={SEED}; Z3={z3.get_version_string()}")
print(
    f"Packages: {len(repo.packages)}; versions: "
    f"{sum(len(package.versions) for package in repo.packages.values())}; "
    f"safe candidates: {len(safe_install_candidates)}"
)

Repository: data/repos/repos/k12/repo_200.json
Group: k12; N=200; seed=42; Z3=5.3.2
Packages: 200; versions: 4891; safe candidates: 155


## Metoda i interpretacja

Generator układa 12 zaplanowanych operacji: 6 instalacji, 3 usunięcia bez autoremove i 3 z autoremove. Kolejność jest losowana seedem, z warunkiem, że usuwanie nie wyprzedzi wszystkich instalacji. Jeśli operacja usuwania zastałaby pusty stan, notebook najpierw wykonuje i raportuje dodatkową instalację — dlatego tabela może mieć więcej niż 12 wierszy.

Do instalacji wybierane są losowo pakiety zależnościowe (`aiohttp`, `h2`, `boto3`) lub pakiety z wersją bez wymagań i konfliktów. Przy usuwaniu losowany jest pakiet ręczny, jeśli taki istnieje, w przeciwnym razie dowolny zainstalowany pakiet.

`Changes` liczy instalacje, usunięcia i zmiany wersji z `build_change_plan`; sama zmiana flagi `manual` jest pokazana osobno i nie zwiększa tej liczby. `Time (s)` obejmuje wywołanie use case’u, bez czasu weryfikacji i przygotowania raportu.

In [2]:
rng = random.Random(SEED)
base_actions = ["install"] * 6 + ["remove-keep"] * 3 + ["remove-auto"] * 3

# Losuj kolejność tak, by przed ostatnią pozycją nie zabrakło instalacji do usunięcia.
for attempt in range(10_000):
    candidate_actions = base_actions.copy()
    rng.shuffle(candidate_actions)
    install_count = 0
    remove_count = 0
    prefix_valid = True
    for index, action in enumerate(candidate_actions):
        if action == "install":
            install_count += 1
        else:
            remove_count += 1
        if index < len(candidate_actions) - 1 and install_count <= remove_count:
            prefix_valid = False
            break
    if prefix_valid:
        actions = candidate_actions
        break
else:
    raise RuntimeError("Could not generate a balanced operation order")

state = State()
assert verify(repo, state.get_packages_list()) == []
records = []

def make_record(step, operation, before, after, elapsed, status):
    errors = verify(repo, after.get_packages_list())
    if errors:
        raise AssertionError(f"State after operation {step} is invalid: {errors}")
    changes = build_change_plan(before, after)
    return {
        "step": step,
        "operation": operation,
        "changes": len(changes),
        "packages": len(after.packages),
        "manual": sum(manual for _, manual in after.packages.values()),
        "time_s": elapsed,
        "verified": "yes",
        "status": status,
    }

def run_install(current_state, package_name, operation):
    before = State(packages=dict(current_state.packages))
    started = time.perf_counter()
    try:
        with redirect_stdout(StringIO()):
            after = install(repo, [package_name], initial_state=current_state)
        status = "ok"
    except PackageError as exc:
        after = before
        status = f"failed: {exc}"
    elapsed = time.perf_counter() - started
    record = make_record(len(records) + 1, operation, before, after, elapsed, status)
    return after, record

for action in actions:
    if action.startswith("remove") and not state.packages:
        seed_package = rng.choice(safe_install_candidates)
        state, record = run_install(
            state, seed_package, f"install (prepare for removal) {seed_package}"
        )
        records.append(record)
        if not state.packages:
            raise RuntimeError("Could not seed an environment for a removal")

    if action == "install":
        package_name = (
            rng.choice(DEPENDENCY_ROOTS)
            if rng.random() < 0.5
            else rng.choice(safe_install_candidates)
        )
        state, record = run_install(
            state, package_name, f"install {package_name}"
        )
    else:
        manual_packages = sorted(
            name for name, (_, manual) in state.packages.items() if manual
        )
        installed_names = manual_packages or sorted(state.packages)
        package_name = rng.choice(installed_names)
        before = State(packages=dict(state.packages))
        started = time.perf_counter()
        autoremove = action == "remove-auto"
        try:
            after = remove(repo, state, [package_name], autoremove=autoremove)
            status = "ok"
        except PackageError as exc:
            after = before
            status = f"failed: {exc}"
        elapsed = time.perf_counter() - started
        state = after
        operation = f"remove {package_name} (autoremove={autoremove})"
        record = make_record(
            len(records) + 1, operation, before, state, elapsed, status
        )
    records.append(record)

print(f"Scheduled actions: {len(actions)}; report rows: {len(records)}")

# Convert records to a DataFrame and round time
df = pd.DataFrame(records)
if "time_s" in df.columns:
    df["time_s"] = df["time_s"].round(3)

df


Scheduled actions: 12; report rows: 13


,step,operation,changes,packages,manual,time_s,verified,status
0,1,install aiohttp,10,10,1,3.505,yes,ok
1,2,install multidict,0,10,2,3.182,yes,ok
2,3,install h2,3,13,3,3.336,yes,ok
3,4,install cffi,2,15,4,3.538,yes,ok
4,5,remove cffi (autoremove=False),1,14,3,2.546,yes,ok
5,6,install h2,0,14,3,3.295,yes,ok
6,7,remove multidict (autoremove=True),11,3,1,2.549,yes,ok
7,8,remove h2 (autoremove=True),3,0,0,2.548,yes,ok
8,9,install (prepare for removal) loguru,1,1,1,3.453,yes,ok
9,10,remove loguru (autoremove=True),1,0,0,2.563,yes,ok


Wszystkie 13 stanów pośrednich zostało zweryfikowanych procedurą z zadania 4.4. Dodatkowa instalacja w raporcie powstała, bo wcześniejsze autoremove opróżniło stan przed kolejną zaplanowaną operacją usuwania. Czas jest mierzony dla samego use case’u; wynik tabeli może mieć inne czasy przy kolejnym uruchomieniu.